# Flappy Bird Reinforcement Learning: Analysis and Results

This notebook documents the **problem formulation**, **mathematical setup**, **method**, and **results** of training a PPO (Proximal Policy Optimization) agent to play Flappy Bird. The codebase lives in the parent directory; here we focus on the math, the environment design, and the analysis of training outcomes.

## 1. Problem formulation and significance

**Problem:** Can an agent learn to play Flappy Bird—avoiding pipes and maximising the number of pipes passed—using only a stream of observations and scalar rewards?

**Why RL?** The game is sequential: each flap or no-flap decision affects future states. We do not have a perfect model of physics or pipe positions in closed form; we have a simulator (the game). Reinforcement learning is well-suited to such **sequential decision-making under uncertainty**: we treat the game as a **Markov decision process (MDP)** and learn a policy that maximises cumulative reward.

**Scope:** We use a single agent (PPO), one reward design, and one observation encoding. The objective is to demonstrate a complete pipeline (environment → training → evaluation) and to analyse learning progress.

## 2. Mathematical setup

### 2.1 Markov decision process

We model Flappy Bird as an MDP with:

- **State space** $\mathcal{S}$: the observation vector (see below), which we treat as a sufficient summary of the current situation.
- **Action space** $\mathcal{A} = \{0, 1\}$: *do nothing* (0) or *flap* (1).
- **Transition dynamics** $P(s' \mid s, a)$: given by the game physics and pipe generation (not written in closed form).
- **Reward** $r_t = R(s_t, a_t, s_{t+1})$: defined below.
- **Discount** $\gamma \in (0, 1]$: we use $\gamma = 0.99$ so the agent values long-term survival.

The **return** (discounted sum of rewards) for a trajectory is

$$
G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k}.
$$

The goal is to find a **policy** $\pi(a \mid s)$ that maximises the **expected return** $\mathbb{E}_{\pi}[G_0]$ from the initial state.

### 2.2 Reward design

Rewards are given at each time step:

$$
r_t = \begin{cases}
+0.1 & \text{step alive} \\
+10   & \text{pipe passed} \\
-100  & \text{death}
\end{cases}
$$

This **reward shaping** encourages the agent to stay alive (small positive per step), strongly rewards passing pipes, and penalises collision. The scale is chosen so that learning is driven by both survival and pipe-passing.

### 2.3 Observation space

The state is a **6-dimensional vector** (normalised to roughly $[-1, 1]$):

| Index | Meaning | Normalisation |
|-------|--------|----------------|
| 0 | Bird height | $y / H$ (screen height $H$) |
| 1 | Bird vertical velocity | clipped $v / v_{\max}$ |
| 2 | Horizontal distance to next pipe | $(x_{\text{pipe}} - x_{\text{bird}}) / W$ |
| 3 | Vertical distance to gap centre | $(y_{\text{gap}} - y_{\text{bird}}) / H$ |
| 4 | Distance to top pipe bottom | normalised |
| 5 | Distance to bottom pipe top | normalised |

So the agent sees *where it is*, *how it is moving*, and *where the next gap is*, which is a minimal sufficient representation for a reactive policy.

### 2.4 PPO (high level)

**Proximal Policy Optimization** (PPO) is a policy-gradient method that updates a neural network policy $\pi_\theta(a\mid s)$ using rollouts from the environment. The objective combines:

1. **Policy gradient** on the advantage estimate $\hat{A}_t$ (e.g. GAE):
   $$
   L^{\text{PG}}(\theta) = \mathbb{E}_t\bigl[ \log \pi_\theta(a_t\mid s_t) \, \hat{A}_t \bigr].
   $$

2. **Clipping** to limit how much the policy changes per update, improving stability:
   $$
   L^{\text{CLIP}}(\theta) = \mathbb{E}_t\bigl[ \min\bigl( r_t(\theta)\,\hat{A}_t,\; \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\,\hat{A}_t \bigr) \bigr],
   $$
   where $r_t(\theta) = \pi_\theta(a_t\mid s_t) / \pi_{\theta_{\text{old}}}(a_t\mid s_t)$.

3. **Value loss** and **entropy bonus** to encourage exploration and accurate value estimates.

We do not re-derive PPO here; we use the Stable-Baselines3 implementation with default hyperparameters (e.g. $\gamma = 0.99$, learning rate $3\times10^{-4}$, 2048 steps per rollout).

## 3. Environment and code (from the repo)

The project structure:

- **`game/`** – Flappy Bird in Pygame (physics, collision, score).
- **`env/`** – Gymnasium wrapper: `FlappyBirdEnv` exposes `step`, `reset`, and the 6-D observation.
- **`training/`** – PPO training script; saves the model as `flappy_model.zip`.
- **`training_stats/`** – Callback that records score per episode and saves JSON/CSV/PNG.

## 4. Training results (1M timesteps)

Score vs Episode for a **1 million timestep** training run:

![Score vs Episode (1M timesteps)](stats-1mil/flappy_training_1000000.png)

**Conclusion and interpretation:**

- **Initial plateau (episodes 0–~700):** The agent rarely passes a single pipe; score and moving average stay near zero. The reward signal is sparse (almost every episode returns about −92.7), so the policy gets little gradient toward "flap at the right time" until it occasionally gets lucky.

- **Breakthrough (~700–850):** The moving average rises sharply as the agent starts passing pipes more often. Once a few successful trajectories appear, PPO reinforces them and the policy improves quickly.

- **High variance / plateau (~850+):** The agent can achieve very high scores (e.g. 700–2500) in some episodes, but the moving average levels off or dips. Performance is inconsistent—the policy has learned good behaviour but still explores and sometimes fails early. Further training or tuning (e.g. reward scale, exploration) could improve stability.

## 5. Conclusion and next steps

We formulated Flappy Bird as an MDP, defined a 6-D observation and a shaped reward, and trained a PPO policy using the Stable-Baselines3 implementation. The **Score vs Episode** curve and summary statistics show whether the agent learned to pass pipes and survive longer.

**Possible next steps:**

- Train for more timesteps (e.g. 1M+) to improve stability.
- Tune reward scale or add sparse rewards only on pipe pass / death.
- Try other algorithms (e.g. DQN, A2C) for comparison.
- Add a time limit per episode to avoid excessively long runs.

**References and resources used in this project:**

- Schulman et al., [*Proximal Policy Optimization Algorithms*](https://arxiv.org/abs/1707.06347) (2017).
- [Stable-Baselines3](https://stable-baselines3.readthedocs.io/) (PPO and Gymnasium integration).
- [Gymnasium](https://gymnasium.farama.org/) – RL environment API.
- [Pygame documentation](https://www.pygame.org/docs/) – game framework.
- [Flappy Bird with Pygame – YouTube](https://youtu.be/hyKL58CySV0?si=zF3BJG4sXJgyxIwb) – tutorial/reference.
- [Reinforcement learning / PPO – YouTube](https://youtu.be/VnpRp7ZglfA?si=-nVTGRvFgEMHN5C-) – tutorial/reference.
- [Flappy Bird assets](https://kosresetr55.itch.io/flappy-bird-assets-by-kosresetr55) by Kosresetr55 – game assets.
- [IBM: Proximal Policy Optimization](https://www.ibm.com/think/topics/proximal-policy-optimization) – PPO overview.